/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: /usr/local/lib/python3.10/dist-packages/tensorflow/python/platform/../../libtensorflow_cc.so.2: undefined symbol: ncclCommRegister

In [1]:
import torch
import torch.nn as nn
from torchvision import models, transforms

# 重新加载模型结构
model = models.vit_b_16(weights=None)
model.heads = nn.Sequential(nn.Linear(model.hidden_dim, 2))

# 加载训练好的权重
model.load_state_dict(torch.load("./best_model.pth", map_location="cpu"))

# 保存完整模型（包含配置信息）
torch.save({
    'model_state_dict': model.state_dict(),
    'num_classes': 2,
    'class_names': ['flooded or damaged buildings', 'undamaged buildings'],
    'preprocess': transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ]),
}, "./hurricane_model_complete.pth")

print("✅ 完整模型已保存: hurricane_model_complete.pth")

/tmp/ipykernel_32369/2188428030.py:10: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("./best_model.pth", map_location="cpu"))


✅ 完整模型已保存: hurricane_model_complete.pth


In [2]:
"""
飓风图像分类 - 推理脚本
使用训练好的模型进行预测
"""

import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os

def load_model(checkpoint_path="./best_model.pth", num_classes=2):
    """加载训练好的模型"""
    model = models.vit_b_16(weights=None)
    model.heads = nn.Sequential(nn.Linear(model.hidden_dim, num_classes))
    model.load_state_dict(torch.load(checkpoint_path, map_location="cpu"))
    model.eval()
    return model

def predict(image_path, model=None, class_names=None):
    """对单张图片进行预测"""
    if model is None:
        model = load_model()
    
    if class_names is None:
        class_names = ['flooded or damaged buildings', 'undamaged buildings']
    
    preprocess = transforms.Compose([
        transforms.Resize(224),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])
    
    image = Image.open(image_path).convert("RGB")
    input_tensor = preprocess(image).unsqueeze(0)
    
    with torch.no_grad():
        output = model(input_tensor)
        probs = torch.softmax(output, dim=1)
        pred = output.argmax(dim=1).item()
        confidence = probs[0][pred].item()
    
    return {
        "prediction": class_names[pred],
        "confidence": f"{confidence:.4f}",
        "class_id": pred
    }

# ========== 测试推理 ==========
if __name__ == "__main__":
    # 检查当前目录下有没有图片可以测试
    current_dir = os.listdir(".")
    image_files = [f for f in current_dir if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
    
    if image_files:
        test_image = image_files[0]
        print(f"使用测试图片: {test_image}")
        result = predict(test_image)
        print(f"预测结果: {result}")
    else:
        print("当前目录没有测试图片，请提供图片路径")
        print("用法: result = predict('your_image.jpg')")
        
        # 演示用随机张量测试模型是否能正常推理
        print("\n使用随机张量测试模型...")
        model = load_model()
        dummy_input = torch.randn(1, 3, 224, 224)
        with torch.no_grad():
            output = model(dummy_input)
            pred = output.argmax(dim=1).item()
        print(f"随机输入测试通过，输出形状: {output.shape}, 预测类别: {pred}")

当前目录没有测试图片，请提供图片路径
用法: result = predict('your_image.jpg')

使用随机张量测试模型...


/tmp/ipykernel_32369/2093836495.py:16: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(checkpoint_path, map_location="cpu"))


随机输入测试通过，输出形状: torch.Size([1, 2]), 预测类别: 0


In [3]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models, transforms
from datasets import load_dataset
from sklearn.metrics import accuracy_score

print(f"PyTorch: {torch.__version__}")

# 加载数据集
dataset = load_dataset("jonathan-roberts1/Satellite-Images-of-Hurricane-Damage")
dataset_split = dataset["train"].train_test_split(test_size=0.2, seed=42)
val_test_split = dataset_split["test"].train_test_split(test_size=0.5, seed=42)

# 只取500张训练，100张验证
train_data = dataset_split["train"].shuffle(seed=42).select(range(500))
val_data = val_test_split["train"].shuffle(seed=42).select(range(100))

# 预处理
preprocess = transforms.Compose([
    transforms.Resize(224),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

def transform_dataset(examples):
    images = [preprocess(img.convert("RGB")) for img in examples["image"]]
    return {"pixel_values": images, "labels": examples["label"]}

train_data = train_data.with_transform(transform_dataset)
val_data = val_data.with_transform(transform_dataset)

def collate_fn(batch):
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    labels = torch.tensor([item["labels"] for item in batch])
    return {"pixel_values": pixel_values, "labels": labels}

train_loader = DataLoader(train_data, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_data, batch_size=8, collate_fn=collate_fn)

# 模型
model = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
model.heads = nn.Sequential(nn.Linear(model.hidden_dim, 2))
device = torch.device("cpu")
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-4)

# 训练2轮，记录真实数据
for epoch in range(2):
    model.train()
    total_loss = 0
    for batch in train_loader:
        pixel_values = batch["pixel_values"].to(device)
        labels = batch["labels"].to(device)
        
        optimizer.zero_grad()
        outputs = model(pixel_values)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    
    # 验证
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            outputs = model(batch["pixel_values"].to(device))
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch["labels"].numpy())
    
    val_acc = accuracy_score(all_labels, all_preds)
    print(f"Epoch {epoch+1}/2, Loss: {avg_loss:.4f}, Val Acc: {val_acc:.4f}")

# 保存
torch.save(model.state_dict(), "./best_model_quick.pth")
print(f"\n✅ 训练完成！验证准确率: {val_acc:.4f}")

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch: 2.4.1
Epoch 1/2, Loss: 0.7914, Val Acc: 0.5100
Epoch 2/2, Loss: 0.6015, Val Acc: 0.7100

✅ 训练完成！验证准确率: 0.7100
